# Phase 2: Qwen2-VL-7B QLoRA 파인튜닝

## 목표
- CARLA 자율주행 QA 데이터셋으로 Qwen2-VL-7B 파인튜닝
- 4-bit QLoRA: RTX 4080 Super 16GB에서 7B 모델 학습
- 파인튜닝 전/후 응답 품질 비교

## 핵심 개념
- **QLoRA**: Quantized LoRA — 4-bit 양자화 + Low-Rank Adaptation
- **LoRA**: W = W₀ + BA (B∈R^{d×r}, A∈R^{r×k}, rank r≪min(d,k))
- **NF4**: Normal Float 4-bit, 가우시안 분포에 최적화된 양자화
- **Target modules**: Attention (q,k,v,o) + FFN (gate, up, down) proj


In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

## 0. 패키지 설치

In [ ]:
# 필요 패키지 버전 확인
# 이미 설치되어 있지 않으면 아래 주석 해제 후 실행
# !pip install trl qwen-vl-utils "peft>=0.18.0" -q

import importlib
for pkg in ['trl', 'qwen_vl_utils', 'peft', 'transformers', 'bitsandbytes', 'accelerate']:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'OK')
        print(f'[OK] {pkg} {ver}')
    except ImportError:
        print(f'[MISSING] {pkg} — pip install {pkg} 실행 필요')


## 1. 임포트 및 환경 확인

In [ ]:
import os, json, random
from pathlib import Path
from typing import List, Dict, Optional
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

import torch
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import Dataset, DataLoader

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. 하이퍼파라미터 설정

In [ ]:
# ── 경로 ──────────────────────────────────────────────
PROJECT_ROOT = Path('C:/Users/apple/Desktop/autonomous_cv_pipeline')
VLM_DIR      = PROJECT_ROOT / 'vlm_driving'
DATA_DIR     = VLM_DIR / 'qa_dataset'
OUTPUT_DIR   = VLM_DIR / 'checkpoints'
OUTPUT_DIR.mkdir(exist_ok=True)

# ── 모델 ──────────────────────────────────────────────
# 2B: 빠른 검증용 (~4GB 다운로드, 학습 ~30분/epoch)
# 7B: 포트폴리오 최종용 (~15GB 다운로드, 학습 ~90분/epoch)
MODEL_NAME = 'Qwen/Qwen2-VL-2B-Instruct'   # 빠른 검증
# MODEL_NAME = 'Qwen/Qwen2-VL-7B-Instruct'  # 최종 제출용

# ── QLoRA 설정 ────────────────────────────────────────
LORA_RANK     = 16     # rank (작을수록 적은 파라미터)
LORA_ALPHA    = 32     # scaling = alpha/rank = 2 (보통 rank*2)
LORA_DROPOUT  = 0.05
# Qwen2-VL attention + FFN projection layers
LORA_TARGETS  = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                 'gate_proj', 'up_proj', 'down_proj']

# ── 학습 설정 ─────────────────────────────────────────
BATCH_SIZE      = 1      # 7B 모델 + 이미지 → 1로 고정
GRAD_ACCUM      = 8      # 유효 배치 = 1×8 = 8
LEARNING_RATE   = 2e-4
NUM_EPOCHS      = 3
MAX_SEQ_LEN     = 512    # 최대 토큰 수 (이미지 포함)
WARMUP_RATIO    = 0.03
SAVE_STEPS      = 100
LOGGING_STEPS   = 10

# ── 이미지 크기 제한 (VRAM 절약) ──────────────────────
# Qwen2-VL: min/max pixels로 동적 해상도 조절
MIN_PIXELS = 256 * 28 * 28   # ~200k pixels
MAX_PIXELS = 512 * 28 * 28   # ~400k pixels

print('설정 완료')
print(f'  유효 배치 크기: {BATCH_SIZE * GRAD_ACCUM}')
print(f'  LoRA 파라미터 수: rank={LORA_RANK}, alpha={LORA_ALPHA}')
print(f'  최대 시퀀스 길이: {MAX_SEQ_LEN}')

## 3. 데이터 로드 및 분포 확인

In [ ]:
with open(DATA_DIR / 'train.json', encoding='utf-8') as f:
    train_data = json.load(f)
with open(DATA_DIR / 'val.json', encoding='utf-8') as f:
    val_data = json.load(f)

print(f'Train: {len(train_data):,}개')
print(f'Val:   {len(val_data):,}개')
print()

# QA 유형 분포
from collections import Counter
train_types = Counter(d['metadata']['qa_type'] for d in train_data)
print('=== Train QA 유형 분포 ===')
for k, v in sorted(train_types.items()):
    print(f'  {k:25s}: {v:4d}개  ({v/len(train_data)*100:.1f}%)')

In [ ]:
# 파인튜닝 전 샘플 질문 저장 (나중에 비교용)
BEFORE_FINETUNE_SAMPLES = [
    train_data[0],    # scene_description
    train_data[425],  # danger_assessment
    train_data[850],  # action_recommendation
]

print('=== 파인튜닝 전 비교 샘플 ===\n')
for s in BEFORE_FINETUNE_SAMPLES:
    q = s['conversations'][0]['value'].replace('<image>\n', '')
    a = s['conversations'][1]['value']
    print(f"[{s['metadata']['qa_type']}]")
    print(f'Q: {q}')
    print(f'정답: {a}')
    print()

## 4. 모델 로드 (4-bit QLoRA)

In [ ]:
# 4-bit 양자화 설정
# - NF4 (Normal Float 4): 가우시안 분포 가중치에 최적화
# - double quant: 양자화 상수도 양자화 → 추가 메모리 절약
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

print(f'{MODEL_NAME} 로딩 중... (최초 실행 시 ~15GB 다운로드)')
print('로딩 후 4-bit 양자화 적용 → ~4-5GB VRAM 예상')

In [ ]:
# Processor 먼저 로드 (경량, 빠름)
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
print(f'Processor 로드 완료')
print(f'  vocab size: {processor.tokenizer.vocab_size:,}')

In [ ]:
# 모델 로드 (4-bit)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)
print(f'모델 로드 완료')

# 파라미터 수 확인
total = sum(p.numel() for p in model.parameters())
print(f'총 파라미터: {total/1e9:.2f}B')

# VRAM 사용량
if torch.cuda.is_available():
    vram_used = torch.cuda.memory_allocated() / 1e9
    print(f'VRAM 사용: {vram_used:.1f} GB')

## 5. LoRA 적용

### LoRA 수식
기존 weight W₀ ∈ R^{d×k} 를 freeze하고 저랭크 행렬 쌍을 학습:

$$h = W_0 x + \frac{\alpha}{r} B A x$$

- A ∈ R^{r×k} : 가우시안 초기화
- B ∈ R^{d×r} : 0 초기화 (초기에는 ΔW=0)
- 학습 파라미터 수: 2×r×(d+k) ≪ d×k

In [ ]:
# kbit training을 위한 준비 (gradient checkpointing 등)
model = prepare_model_for_kbit_training(model)

# LoRA 설정
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 학습 가능 파라미터 수 직접 계산
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'학습 가능: {trainable/1e6:.1f}M / 전체: {total/1e9:.2f}B ({trainable/total*100:.2f}%)')

## 6. 데이터셋 클래스

Qwen2-VL 입력 포맷:
```
<|im_start|>system
...<|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>
질문<|im_end|>
<|im_start|>assistant
답변<|im_end|>
```

Loss는 assistant 답변 토큰에만 계산 (질문 부분은 -100으로 마스킹)

In [ ]:
SYSTEM_PROMPT = (
    '당신은 자율주행 차량의 카메라 영상을 분석하는 전문 AI입니다. '
    '주어진 이미지에서 도로 상황, 주변 객체, 위험 요소를 정확하게 파악하고 '
    '안전 운전을 위한 정보를 제공합니다.'
)

class CarlaVQADataset(Dataset):
    # Qwen2-VL용 데이터셋
    # Dataset 레벨에서 padding/truncation 안 함 (이미지 토큰 수 동적)
    # collate_fn에서 배치 내 최대 길이로 패딩
    def __init__(self, data, processor):
        self.data = data
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(item['image']).convert('RGB')
        question = item['conversations'][0]['value'].replace('<image>', '').strip()
        answer   = item['conversations'][1]['value'].strip()

        # 학습용 (질문+답변 포함)
        messages_full = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': image},
                    {'type': 'text',  'text': question},
                ],
            },
            {'role': 'assistant', 'content': answer},
        ]
        text_full = self.processor.apply_chat_template(
            messages_full, tokenize=False, add_generation_prompt=False
        )

        # 질문만 (loss 마스킹 경계 탐색)
        messages_q = messages_full[:-1]
        text_q = self.processor.apply_chat_template(
            messages_q, tokenize=False, add_generation_prompt=True
        )

        # 인코딩 (padding/truncation 없이) — processor가 반환하는 모든 키 보존
        inputs = self.processor(
            text=[text_full],
            images=[image],
            return_tensors='pt',
        )
        inputs_q = self.processor(
            text=[text_q],
            images=[image],
            return_tensors='pt',
        )

        input_ids      = inputs['input_ids'].squeeze(0)
        attention_mask = inputs['attention_mask'].squeeze(0)
        question_len   = inputs_q['input_ids'].shape[1]

        # labels: 질문/이미지 토큰 -100 마스킹
        labels = input_ids.clone()
        labels[:question_len] = -100

        result = {
            'input_ids':      input_ids,
            'attention_mask': attention_mask,
            'labels':         labels,
        }

        # Qwen2-VL 필수 필드 전부 포함
        for key in ['pixel_values', 'image_grid_thw', 'mm_token_type_ids',
                    'video_grid_thw', 'second_per_grid_ts']:
            if key in inputs:
                val = inputs[key]
                result[key] = val.squeeze(0) if val.dim() > 1 and val.shape[0] == 1 else val

        return result


def collate_fn(batch):
    # 동적 패딩: 배치 내 최대 길이로 패딩
    pad_id = 0
    max_len = max(b['input_ids'].shape[0] for b in batch)

    def pad_1d(seqs, pad_val):
        out = []
        for s in seqs:
            pad_len = max_len - s.shape[0]
            out.append(torch.cat([s, torch.full((pad_len,), pad_val, dtype=s.dtype)]))
        return torch.stack(out)

    result = {
        'input_ids':      pad_1d([b['input_ids'] for b in batch], pad_id),
        'attention_mask': pad_1d([b['attention_mask'] for b in batch], 0),
        'labels':         pad_1d([b['labels'] for b in batch], -100),
    }

    # mm_token_type_ids: input_ids와 동일한 shape → 같은 방식으로 패딩 (pad값=0)
    if 'mm_token_type_ids' in batch[0]:
        result['mm_token_type_ids'] = pad_1d([b['mm_token_type_ids'] for b in batch], 0)

    # pixel_values: (num_patches, C*kernel*kernel) — 배치 내 concat
    if 'pixel_values' in batch[0]:
        pvs = [b['pixel_values'] for b in batch]
        result['pixel_values'] = torch.cat(pvs, dim=0) if pvs[0].dim() == 2 else torch.stack(pvs)

    # image_grid_thw: (num_images, 3) — 배치 내 concat
    if 'image_grid_thw' in batch[0]:
        thws = [b['image_grid_thw'] for b in batch]
        if thws[0].dim() == 1:
            result['image_grid_thw'] = torch.stack(thws)
        else:
            result['image_grid_thw'] = torch.cat(thws, dim=0)

    return result


# 데이터셋 생성
train_dataset = CarlaVQADataset(train_data, processor)
val_dataset   = CarlaVQADataset(val_data,   processor)

print(f'Train dataset: {len(train_dataset):,}개')
print(f'Val   dataset: {len(val_dataset):,}개')

# 샘플 키 확인
sample = train_dataset[0]
print('샘플 키:', list(sample.keys()))
for k, v in sample.items():
    print(f'  {k}: {v.shape}')


In [ ]:
# 데이터셋 샘플 테스트
print('데이터셋 샘플 로드 테스트...')
sample = train_dataset[0]
print(f'  input_ids shape:      {sample["input_ids"].shape}')
print(f'  attention_mask shape: {sample["attention_mask"].shape}')
print(f'  labels shape:         {sample["labels"].shape}')

if 'pixel_values' in sample:
    print(f'  pixel_values shape:   {sample["pixel_values"].shape}')
if 'image_grid_thw' in sample:
    print(f'  image_grid_thw:       {sample["image_grid_thw"]}')

# label 마스킹 확인
labels = sample['labels']
unmasked = (labels != -100).sum().item()
total_toks = (labels != processor.tokenizer.pad_token_id).sum().item()
print(f'  Loss 계산 토큰: {unmasked} / 전체: {labels.shape[0]}')
print('샘플 테스트 통과!')

## 7. 파인튜닝 전 베이스라인 추론

파인튜닝 후와 비교할 기준점 확인

In [ ]:
def inference(model, processor, image_path: str, question: str,
              max_new_tokens: int = 256) -> str:
    """단일 이미지 + 질문 → 답변 생성"""
    image = Image.open(image_path).convert('RGB')
    
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': image},
                {'type': 'text',  'text': question},
            ]
        },
    ]
    
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text], images=[image], return_tensors='pt'
    ).to(model.device)
    
    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
        )
    
    # 새로 생성된 토큰만 추출
    generated = output_ids[0][inputs['input_ids'].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True)


print('=== 파인튜닝 전 베이스라인 ===\n')
baseline_results = []

for s in BEFORE_FINETUNE_SAMPLES:
    q    = s['conversations'][0]['value'].replace('<image>\n', '').strip()
    gt   = s['conversations'][1]['value'].strip()
    pred = inference(model, processor, s['image'], q)
    
    baseline_results.append({'qa_type': s['metadata']['qa_type'],
                              'question': q, 'gt': gt, 'pred': pred})
    
    print(f"[{s['metadata']['qa_type']}]")
    print(f'Q: {q}')
    print(f'정답: {gt}')
    print(f'모델: {pred}')
    print()

## 8. 학습 (QLoRA 파인튜닝)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    # 학습 설정
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    # Optimizer
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    optim='paged_adamw_8bit',

    # 정밀도
    fp16=True,
    bf16=False,

    # Gradient checkpointing (메모리 절약)
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},

    # 로깅/저장 (transformers 5.x: evaluation_strategy → eval_strategy)
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=SAVE_STEPS,
    eval_strategy='steps',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,

    # 기타
    dataloader_num_workers=0,
    remove_unused_columns=False,
    report_to='none',
    seed=42,
)

print('학습 설정 완료')
print(f'  총 스텝 수 (예상): {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS:,}')


In [ ]:
# Trainer 커스텀: pixel_values 포함 배치 처리
class VLMTrainer(Trainer):
    """Qwen2-VL 전용 Trainer — pixel_values를 올바르게 전달"""
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss


trainer = VLMTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
)

print('Trainer 초기화 완료')

In [ ]:
# ── 학습 시작 ──────────────────────────────────────────
# 예상 시간: RTX 4080 Super 기준 epoch당 약 20~30분
# 총 3 epoch = 1~1.5시간

print('학습 시작...')
print(f'Train: {len(train_dataset):,}개, Val: {len(val_dataset):,}개')
print(f'유효 배치: {BATCH_SIZE * GRAD_ACCUM}, Epochs: {NUM_EPOCHS}')
print()

train_result = trainer.train()

print('\n=== 학습 완료 ===')
print(f'최종 train loss: {train_result.training_loss:.4f}')
print(f'총 학습 시간: {train_result.metrics["train_runtime"]:.0f}초')

## 9. 학습 곡선 시각화

In [ ]:
# 학습 로그 파싱
logs = trainer.state.log_history

train_steps  = [l['step'] for l in logs if 'loss' in l and 'eval_loss' not in l]
train_losses = [l['loss'] for l in logs if 'loss' in l and 'eval_loss' not in l]
eval_steps   = [l['step'] for l in logs if 'eval_loss' in l]
eval_losses  = [l['eval_loss'] for l in logs if 'eval_loss' in l]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(train_steps, train_losses, label='Train Loss', alpha=0.7, linewidth=1)
ax.plot(eval_steps, eval_losses,   label='Val Loss',   alpha=0.9, linewidth=2, marker='o')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Qwen2-VL-7B QLoRA 학습 곡선 (CARLA 자율주행 QA)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(VLM_DIR / 'training_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

if eval_losses:
    print(f'최초 Val Loss: {eval_losses[0]:.4f}')
    print(f'최종 Val Loss: {eval_losses[-1]:.4f}')
    print(f'개선: {(eval_losses[0] - eval_losses[-1]) / eval_losses[0] * 100:.1f}%')

## 10. 어댑터 저장

In [ ]:
ADAPTER_PATH = VLM_DIR / 'lora_adapter'

# LoRA 어댑터만 저장 (전체 모델이 아닌 delta weights만)
model.save_pretrained(str(ADAPTER_PATH))
processor.save_pretrained(str(ADAPTER_PATH))

# 저장 크기 확인
import os
total_size = sum(
    f.stat().st_size for f in ADAPTER_PATH.rglob('*') if f.is_file()
)
print(f'어댑터 저장 완료: {ADAPTER_PATH}')
print(f'저장 크기: {total_size / 1e6:.1f} MB')
print(f'(LoRA rank={LORA_RANK}: 전체 7B 모델 대비 극소량만 저장)')

# 저장된 파일 목록
for f in sorted(ADAPTER_PATH.iterdir()):
    size = f.stat().st_size / 1e6
    print(f'  {f.name}: {size:.1f} MB')

## 11. 파인튜닝 후 추론 비교

In [ ]:
print('=== 파인튜닝 전후 비교 ===\n')

model.eval()
comparison_rows = []

for i, s in enumerate(BEFORE_FINETUNE_SAMPLES):
    q    = s['conversations'][0]['value'].replace('<image>\n', '').strip()
    gt   = s['conversations'][1]['value'].strip()
    pred = inference(model, processor, s['image'], q)
    
    comparison_rows.append({
        'qa_type':  s['metadata']['qa_type'],
        'question': q,
        'gt':       gt,
        'before':   baseline_results[i]['pred'],
        'after':    pred,
    })
    
    print(f"[{s['metadata']['qa_type']}]")
    print(f'Q:      {q}')
    print(f'정답:   {gt}')
    print(f'이전:   {baseline_results[i]["pred"]}')
    print(f'이후:   {pred}')
    print()

In [ ]:
# 결과 JSON 저장 (evaluation 노트북에서 활용)
results_path = VLM_DIR / 'finetune_comparison.json'
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(comparison_rows, f, ensure_ascii=False, indent=2)

print(f'비교 결과 저장: {results_path}')

## 12. 전체 Val 셋 자동 평가 (선택적)

빠른 정성적 평가: Val 셋 일부(50개)에 대해 키워드 매칭 정확도 측정

In [ ]:
import re

def keyword_accuracy(pred: str, gt: str) -> float:
    """
    정성적 지표: GT의 핵심 키워드가 pred에 포함되는지 확인
    - 숫자 (e.g. 30.6m, 23%) → ±20% 허용
    - 키워드 (차량, 보행자 등) → exact match
    """
    # 숫자 추출
    gt_nums   = [float(x) for x in re.findall(r'\d+\.?\d*', gt)]
    pred_nums = [float(x) for x in re.findall(r'\d+\.?\d*', pred)]
    
    num_score = 0.0
    if gt_nums:
        matches = 0
        for gn in gt_nums:
            for pn in pred_nums:
                if gn != 0 and abs(gn - pn) / abs(gn) < 0.2:
                    matches += 1
                    break
        num_score = matches / len(gt_nums)
    
    # 핵심 키워드 (한국어)
    keywords = ['차량', '보행자', '안전', '위험', '정지', '주행', '차선', '신호']
    gt_kws   = [kw for kw in keywords if kw in gt]
    
    kw_score = 0.0
    if gt_kws:
        kw_score = sum(1 for kw in gt_kws if kw in pred) / len(gt_kws)
    
    # 가중 평균
    if gt_nums and gt_kws:
        return 0.5 * num_score + 0.5 * kw_score
    elif gt_nums:
        return num_score
    elif gt_kws:
        return kw_score
    else:
        return 1.0 if len(pred) > 10 else 0.0  # 응답 있으면 OK


# Val 50개 평가
N_EVAL = 50
eval_samples = random.sample(val_data, min(N_EVAL, len(val_data)))

print(f'Val {N_EVAL}개 평가 중...')
scores_by_type = {}

for i, s in enumerate(eval_samples):
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{N_EVAL}...')
    
    q  = s['conversations'][0]['value'].replace('<image>\n', '').strip()
    gt = s['conversations'][1]['value'].strip()
    qa_type = s['metadata']['qa_type']
    
    pred = inference(model, processor, s['image'], q)
    score = keyword_accuracy(pred, gt)
    
    if qa_type not in scores_by_type:
        scores_by_type[qa_type] = []
    scores_by_type[qa_type].append(score)

print('\n=== QA 유형별 정확도 ===\n')
all_scores = []
for qa_type, scores in sorted(scores_by_type.items()):
    avg = np.mean(scores)
    all_scores.extend(scores)
    print(f'  {qa_type:25s}: {avg:.3f}  (n={len(scores)})')

print(f'\n전체 평균 정확도: {np.mean(all_scores):.3f}')

In [ ]:
# QA 유형별 정확도 바 차트
types_sorted = sorted(scores_by_type.keys())
avgs = [np.mean(scores_by_type[t]) for t in types_sorted]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(types_sorted)), avgs,
              color=['#2196F3' if a >= 0.7 else '#FF9800' if a >= 0.5 else '#F44336'
                     for a in avgs])
ax.set_xticks(range(len(types_sorted)))
ax.set_xticklabels(types_sorted, rotation=20, ha='right')
ax.set_ylabel('정확도 (키워드 매칭)')
ax.set_title('파인튜닝 후 QA 유형별 정확도')
ax.set_ylim(0, 1.1)
ax.axhline(y=np.mean(all_scores), color='red', linestyle='--', label=f'평균: {np.mean(all_scores):.3f}')
ax.legend()

for bar, avg in zip(bars, avgs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{avg:.2f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(str(VLM_DIR / 'qa_accuracy_by_type.png'), dpi=150, bbox_inches='tight')
plt.show()

## 13. 결과 요약

In [ ]:
print('=' * 60)
print('   Qwen2-VL-7B QLoRA 파인튜닝 결과 요약')
print('=' * 60)
print()
print('[모델]')
print(f'  Base:        {MODEL_NAME}')
print(f'  LoRA rank:   {LORA_RANK} (alpha={LORA_ALPHA})')
print(f'  학습 파라미터: ~{trainable/1e6:.0f}M / {total/1e9:.1f}B ({trainable/total*100:.2f}%)')
print()
print('[데이터]')
print(f'  Train: {len(train_dataset):,}개, Val: {len(val_dataset):,}개')
print(f'  QA 유형: 7가지 (장면설명/객체/위험/행동/보행자/수량/거리)')
print()
print('[학습]')
print(f'  Epochs:     {NUM_EPOCHS}')
print(f'  유효 배치:  {BATCH_SIZE * GRAD_ACCUM}')
print(f'  LR:         {LEARNING_RATE}')
if eval_losses:
    print(f'  Val Loss:   {eval_losses[0]:.4f} → {eval_losses[-1]:.4f}')
print()
print('[정확도 (키워드 매칭)]')
for qa_type in types_sorted:
    avg = np.mean(scores_by_type[qa_type])
    print(f'  {qa_type:25s}: {avg:.3f}')
print(f'  {"전체 평균":25s}: {np.mean(all_scores):.3f}')
print()
print('[저장]')
print(f'  어댑터: {ADAPTER_PATH}')
print('=' * 60)
print()
print('다음 단계: 03_evaluation.ipynb')
print('  - VQA Accuracy (BLEU / ROUGE / CIDEr)')
print('  - 파인튜닝 전/후 정량적 비교')
print('  - 오류 사례 분석')